# Lecture 1 Demo (45–60 min) — “универсальный протокол эксперимента” (регрессия, scikit-learn)

**Цель демо:** закрепить “общий язык курса” на живом коде:  
`split → CV → Pipeline → model selection → финальный test (один раз) → логирование результатов`.

> Этот ноутбук — **демонстрационный** (не сдаётся).  
> Творческое задание Lab01 будет отдельным и потребует *самостоятельно* создать датасеты/утечки и собрать мини-таблицу.

## Соответствие с Лекцией 1 (слайды)
1) *ML-алгоритм = (H)+(loss)+(fit)+(evaluation)* → мы фокусируемся на **evaluation**.  
2) *fit vs model selection* → показываем `GridSearchCV` только на train.  
3) *Estimator API* → `.fit/.predict/.score`, `cross_val_score`.  
4) *Pipeline как защита от утечек* → сравнение BAD vs GOOD.

---

## 0) Setup (версии, импорты, RMSE совместимость)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

import sklearn
print("sklearn:", sklearn.__version__)

from sklearn.datasets import load_diabetes, make_regression
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_regression
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# RMSE compatibility across sklearn versions
try:
    from sklearn.metrics import root_mean_squared_error as _rmse_fn
    def rmse(y_true, y_pred):
        return float(_rmse_fn(y_true, y_pred))
except Exception:
    def rmse(y_true, y_pred):
        return float(np.sqrt(mean_squared_error(y_true, y_pred)))

import matplotlib.pyplot as plt  # no seaborn

## 1) Данные и split: test используем ровно один раз

In [ ]:
RANDOM_STATE = 42
SEED = 2026

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("X:", X.shape, "train:", X_train.shape, "test:", X_test.shape)

**Что сказать (1–2 мин):**  
- test — “священная корова”: *не трогаем* до финала;  
- всё “обучение/подбор” делаем внутри train.

## 2) Baseline: если модель не бьёт baseline — дальше бессмысленно

In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
pred = baseline.predict(X_test)

print("Test RMSE:", rmse(y_test, pred))
print("Test MAE:", mean_absolute_error(y_test, pred))
print("Test R2:", r2_score(y_test, pred))

## 3) CV на train: зачем нужно и что даёт

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

ridge = Ridge(alpha=1.0)
scores = cross_val_score(ridge, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")
print("CV RMSE mean/std:", -scores.mean(), scores.std())

ridge.fit(X_train, y_train)
pred = ridge.predict(X_test)
print("Final test RMSE (single use):", rmse(y_test, pred))

**Что обсудить (3–4 мин):**  
- CV даёт *распределение* качества (mean/std), а не одно число;  
- CV считает качество модели, обученной на меньшем числе данных (k-1 фолдов).

## 4) Пример “как у Мюллера”: KNN в 2D и почему масштаб ломает соседей (классификация)

Это **toy-пример** ровно для одной идеи:

> KNN опирается на расстояния. Если один признак измерен в “тысячах”, а другой в “единицах”, то без стандартизации
> расстояния (и значит соседи) определяются почти полностью первым признаком — даже если он не несёт информации о классе.

Мы сгенерируем 2D-данные, где классы разделяются в основном по вертикали (второй координате), а по горизонтали есть шум.
Затем “растянем” горизонтальную ось в 50 раз — и посмотрим, как меняется граница решений KNN:
- **raw (плохо):** KNN почти игнорирует полезную вертикальную координату
- **scaled (хорошо):** после `StandardScaler` KNN снова “видит” правильную геометрию

> Это именно тот смысл, который обычно показывают в материалах Мюллера для StandardScaler/KNN.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.RandomState(SEED)

# 2 "вертикально" разделённых кластера: класс определяется по y-координате
Xc, yc = make_blobs(
    n_samples=220,
    centers=[(0.0, -1.0), (0.0, 1.0)],
    cluster_std=0.45,
    random_state=SEED
)

# "Растягиваем" горизонтальную ось: теперь один признак в 50 раз крупнее по масштабу
Xc_stretched = Xc.copy()
Xc_stretched[:, 0] *= 50.0

print("std(feature0)=", Xc_stretched[:,0].std(), " std(feature1)=", Xc_stretched[:,1].std())

In [ ]:
from matplotlib.colors import ListedColormap

def plot_decision_boundary(clf, X, y, title: str):
    # Two-class color palettes without yellow (close to sklearn examples)
    cmap_light = ListedColormap(["#A6CEE3", "#FB9A99"])  # light blue / light red background
    cmap_bold  = ListedColormap(["#1F78B4", "#E31A1C"])  # blue / red points

    # grid
    x_min, x_max = X[:, 0].min() - 1.0, X[:, 0].max() + 1.0
    y_min, y_max = X[:, 1].min() - 1.0, X[:, 1].max() + 1.0
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 400),
        np.linspace(y_min, y_max, 300)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = clf.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(6, 4))
    plt.contourf(xx, yy, Z, alpha=0.25, cmap=cmap_light)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_bold, s=18, alpha=0.9, edgecolors="k", linewidths=0.2)
    plt.xlabel("feature 0 (stretched)")
    plt.ylabel("feature 1")
    plt.title(title)
    plt.show()

In [ ]:
# Сравниваем CV-качество: raw vs scaled
cv2 = KFold(n_splits=5, shuffle=True, random_state=SEED)

knn_clf = KNeighborsClassifier(n_neighbors=15)
scores_raw = cross_val_score(knn_clf, Xc_stretched, yc, cv=cv2, scoring="accuracy")

pipe_clf = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))
scores_scaled = cross_val_score(pipe_clf, Xc_stretched, yc, cv=cv2, scoring="accuracy")

print("KNN accuracy raw   (mean/std):", scores_raw.mean(), scores_raw.std())
print("KNN accuracy scaled(mean/std):", scores_scaled.mean(), scores_scaled.std())

In [ ]:
# Граница решений (raw)
knn_clf.fit(Xc_stretched, yc)
plot_decision_boundary(knn_clf, Xc_stretched, yc, "KNN (raw): feature0 dominates distances → boundary looks wrong")

# Граница решений (scaled via Pipeline)
pipe_clf.fit(Xc_stretched, yc)
plot_decision_boundary(pipe_clf, Xc_stretched, yc, "KNN + StandardScaler (Pipeline): geometry restored → boundary sensible")

**Что проговорить (1–2 минуты):**
- Здесь **не нужен test**: это локальная иллюстрация *почему preprocessing важен* для методов на расстояниях.
- Важный мостик: `StandardScaler` надо делать **внутри CV** → поэтому используем `Pipeline` (и этим же логически переходим к leakage в следующем разделе).

## 5) Leakage: самый показательный пример — feature selection, которая использует y

### Что значит “для каждого признака отдельно считается F-статистика связи с $y$”

`SelectPercentile(score_func=f_regression)` — это **одномерный (univariate) отбор признаков**:
он рассматривает **каждый признак $x_j$ по отдельности** и задаёт вопрос:

> “Есть ли линейная связь между $x_j$ и $y$, если игнорировать остальные признаки?”

Формально для каждого признака $x_j$ мы рассматриваем простую линейную регрессию
$$
y = a + b\,x_j + \varepsilon
$$
и проверяем гипотезу
$$
H_0:\ b = 0 \quad \text{(нет линейной связи)}
$$
против
$$
H_1:\ b \neq 0.
$$

#### Откуда берётся F-статистика
В такой “одномерной” регрессии можно разложить вариацию целевой переменной:
- **SST**: общая сумма квадратов $ \sum_i (y_i-\bar y)^2 $
- **SSR**: объяснённая моделью сумма квадратов $ \sum_i (\hat y_i-\bar y)^2 $
- **SSE**: остаточная сумма квадратов $ \sum_i (y_i-\hat y_i)^2 $

И выполняется: $ \text{SST}=\text{SSR}+\text{SSE} $.

F-статистика сравнивает “объяснённую” и “необъяснённую” части (с поправкой на степени свободы):
$$
F = \frac{\text{SSR}/1}{\text{SSE}/(n-2)}.
$$
Если $F$ большой, это означает: **модель с $x_j$ объясняет заметную долю вариации $y$** по сравнению с шумом.

#### Связь с корреляцией (интуиция)
В одномерном случае F тесно связан с квадратом корреляции $r$ между $x_j$ и $y$:
$$
F = \frac{r^2}{1-r^2}\,(n-2).
$$
То есть “большой F” примерно означает “большая по модулю корреляция с $y$” (в линейном смысле).

#### Почему это важно для leakage
Заметьте: чтобы посчитать $F$ для признака, нужно **использовать $y$**.
Поэтому `select.fit(X, y)` — это обучение преобразования, зависящего от $y$.
Если сделать это до CV (BAD), вы подглядываете в будущие val-fold'ы. В `Pipeline` (GOOD) `fit` происходит заново внутри каждого fold.

In [ ]:
# Делаем датасет, где легко “сломать” протокол
# small n, huge d -> feature selection может сильно переобучиться
Xh, yh = make_regression(
    n_samples=120, n_features=2000, n_informative=20, noise=30.0, random_state=SEED
)
Xh_train, Xh_test, yh_train, yh_test = train_test_split(Xh, yh, test_size=0.2, random_state=RANDOM_STATE)
cvh = KFold(n_splits=5, shuffle=True, random_state=SEED)

select = SelectPercentile(score_func=f_regression, percentile=5)  # uses y in fit!
model = Ridge(alpha=1.0)

# BAD: selection fit on all training data (includes future val folds)
Xh_sel_bad = select.fit_transform(Xh_train, yh_train)
scores_bad = cross_val_score(model, Xh_sel_bad, yh_train, cv=cvh, scoring="neg_root_mean_squared_error")

# GOOD: selection inside CV via Pipeline (refit per fold)
pipe_good = Pipeline([("select", SelectPercentile(score_func=f_regression, percentile=5)),
                      ("model", Ridge(alpha=1.0))])
scores_good = cross_val_score(pipe_good, Xh_train, yh_train, cv=cvh, scoring="neg_root_mean_squared_error")

print("Feature selection leakage demo (CV RMSE):")
print("BAD  mean/std:", -scores_bad.mean(), scores_bad.std())
print("GOOD mean/std:", -scores_good.mean(), scores_good.std())
print("Optimism (GOOD - BAD):", (-scores_good.mean()) - (-scores_bad.mean()))

### Проверка на hold-out test: “плохой CV” обещал слишком хорошо

Важно: здесь **не сравниваем два разных финальных алгоритма** — финальная модель всё равно обучается на всём train.
Мы сравниваем **оценки качества** (BAD-CV vs GOOD-CV) с реальной ошибкой на test.

Если BAD-CV был оптимистичен из-за leakage, то тестовая ошибка окажется существенно хуже, чем BAD-CV обещал.

In [ ]:
# --- Hold-out test check: does BAD-CV optimism materialize? ---

# Финальная модель (обучение на всём train) — корректная процедура:
# 1) fit feature selection на train (использует y_train — это нормально для обучения!)
# 2) transform train и test через тот же селектор
# 3) fit Ridge на преобразованном train
select_final = SelectPercentile(score_func=f_regression, percentile=5)
Xh_train_sel = select_final.fit_transform(Xh_train, yh_train)
Xh_test_sel = select_final.transform(Xh_test)

model_final = Ridge(alpha=1.0)
model_final.fit(Xh_train_sel, yh_train)
pred_test = model_final.predict(Xh_test_sel)

test_rmse_h = rmse(yh_test, pred_test)

bad_cv_rmse = -scores_bad.mean()
good_cv_rmse = -scores_good.mean()

print("Compare estimates vs hold-out test (RMSE):")
print(f"BAD  CV estimate : {bad_cv_rmse:.3f}")
print(f"GOOD CV estimate : {good_cv_rmse:.3f}")
print(f"Hold-out test    : {test_rmse_h:.3f}")
print(f"BAD optimism gap (test - BAD_CV): {test_rmse_h - bad_cv_rmse:.3f}")
print(f"GOOD gap         (test - GOOD_CV): {test_rmse_h - good_cv_rmse:.3f}")

**Что сказать (6–8 мин):**  
- здесь утечка “жёсткая”: `SelectPercentile(...).fit` использует `y` и видит будущие val-фолды;  
- поэтому BAD-CV может быть *сильно* оптимистичным;  
- Pipeline — это “автоматизированная дисциплина”: каждый fold заново учит select на train-fold.

## 6) Model selection: гиперпараметры выбираем только на train (GridSearchCV)

In [ ]:
param_grid = {"alpha": [0.01, 0.1, 1.0, 10.0, 100.0]}
search = GridSearchCV(
    estimator=Ridge(),
    param_grid=param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    refit=True
)

# IMPORTANT: fit only on train
search.fit(X_train, y_train)

best_alpha = search.best_params_["alpha"]
best_cv_rmse = -search.best_score_

best_model = search.best_estimator_
best_model.fit(X_train, y_train)
pred = best_model.predict(X_test)

print("Best alpha:", best_alpha)
print("CV RMSE (selected):", best_cv_rmse)
print("Final test RMSE (single use):", rmse(y_test, pred))

**Что сказать (4–5 мин):**  
- `GridSearchCV` — это “встроенный внешний цикл выбора модели”;  
- test не участвует ни в fit, ни в выборе гиперпараметров;  
- почему иногда test хуже CV: нормально (шум/смещение/случайность).

## 7) Логирование: runs/leaderboard как артефакт курса

In [ ]:
RUNS_PATH = Path("runs.csv")

def append_run(row: dict, path: Path = RUNS_PATH):
    df = pd.DataFrame([row])
    if path.exists():
        prev = pd.read_csv(path)
        out = pd.concat([prev, df], ignore_index=True)
    else:
        out = df
    out.to_csv(path, index=False)
    return out

row = {
    "timestamp": pd.Timestamp.utcnow().isoformat(),
    "seed": SEED,
    "dataset": "diabetes",
    "model": "Ridge",
    "best_alpha": best_alpha,
    "cv_rmse": float(best_cv_rmse),
    "test_rmse": float(rmse(y_test, pred)),
    "notes": "Lecture01 demo: split->CV->GridSearchCV->test once",
}
runs = append_run(row)
runs.tail(3)

**Что сказать (3–4 мин):**  
- лог — это “научный протокол”: без него сравнение моделей превращается в разговоры;  
- `runs.csv` — общий формат на весь семестр (потом добавим модели/метрики/датасеты).

## 8) Финал: мини-чеклист из 4 вопросов (связь со слайдами)

Когда видите новый алгоритм, задайте:

1) **Что параметризуем?** (класс моделей \( \mathcal H \))  
2) **Чем меряем ошибку?** (loss / метрика)  
3) **Как учим?** (closed-form / GD / QP / EM / SGD / …)  
4) **Как честно оцениваем?** (split/CV/test; утечки; selection)

На Lab01 creative вы *сами* создадите 2 датасета с заданными свойствами и воспроизведёте BAD vs GOOD.